# Lab 4: Navier–Stokes flow

[Start Here](../../Start_Here.ipynb) · Previous: [Lab 3: Heat conduction](../03_heat_conduction/Lab_3_Heat_Conduction.ipynb) · Next: [Challenge 1: Wave](../../02_challenges/01_wave/Challenge_1_Wave_Dynamics.ipynb)

Learn a periodic, incompressible flow using the Taylor–Green vortex as a known reference solution.

This planar model is not a weather forecast: it omits thermodynamics, moisture, planetary rotation, and spherical geometry.

The input is $(x,y,t)$ and the outputs are two velocity components and pressure. Initial data and the three PDE residuals train the model; periodic coordinate features enforce spatial periodicity.

Check velocity and pressure separately with `synthetic_velocity_rmse` and `synthetic_pressure_gauge_aligned_rmse`. Flow depends on pressure gradients, so the pressure metric subtracts the spatially constant error at each evaluated time. Raw pressure error and offset are also saved. Older metrics that mix velocity and pressure remain for compatibility; do not treat them as a combined accuracy score.

Read the colorbars when comparing frames. If each panel uses a different scale, matching colors do not imply equal speeds.

## The flow problem

The network predicts horizontal velocity $u$, vertical velocity $v$, and pressure $p$. With constant density and viscosity, the equations are

$$u_x+v_y=0,$$
$$u_t+uu_x+vu_y+p_x/\rho-\nu(u_{xx}+u_{yy})=0,$$
$$v_t+uv_x+vv_y+p_y/\rho-\nu(v_{xx}+v_{yy})=0.$$

The first equation enforces incompressibility; the other two balance momentum.

### Optional weather-data initial condition

The supplied `data_lat.npy` is described by its source as a projected, tiled ERA5 array. Its acquisition date, units, and preprocessing have not been independently verified; see [data provenance](DATA_PROVENANCE.md). It can supply an initial field for this example, but it does not establish forecast accuracy.

![Projection illustration for the optional data example](images/projection.png)

### Background: general conservation equations

The general equations below also allow variable density and energy transport. This Lab trains on only the constant-density continuity and momentum equations above. The energy equation is background.

\begin{equation}
Continuity : \frac{\partial \rho}{\partial t} + \overrightarrow{\nabla}\cdot(\rho\overrightarrow{u})=0 \end{equation}

\begin{equation}
Momentum : \frac{\partial(\rho \overrightarrow{u})}{\partial t} + \overrightarrow{\nabla}\cdot[\rho\overline{\overline{u\otimes u}}] = -\overrightarrow{\nabla p} + \overrightarrow{\nabla}\cdot\overline{\overline{\tau}} + \rho\overrightarrow{f} \end{equation}

\begin{equation}
Energy : \frac{\partial(\rho e)}{\partial t} + \overrightarrow{\nabla}\cdot((\rho e + p)\overrightarrow{u}) = \overrightarrow{\nabla}\cdot(\overline{\overline{\tau}}\cdot\overrightarrow{u}) + \rho\overrightarrow{f}\overrightarrow{u} + \overrightarrow{\nabla}\cdot(\overrightarrow{\dot{q}})+r \end{equation}

### Step 1: Periodic domain

The domain is $x,y\in[-0.720,0.720]$. Sine and cosine input features make the network periodic in both directions. The optional array supplies tiled samples from -0.720 to 0.719.

![Periodic tiling illustration](images/periodicity_conversion.png)

These planar periodic boundaries are a modeling choice, not a representation of a spherical atmosphere.

### Scaling and nondimensionalization

```python
LENGTH = 1.440
LOWER = -0.720
LENGTH_SCALE = 12742000 / LENGTH
TIME_SCALE = 60 * 60 * 60
VELOCITY_SCALE = LENGTH_SCALE / TIME_SCALE
PRESSURE_SCALE = 1.1614 * VELOCITY_SCALE**2
LEGACY_PRESSURE_FACTOR = 0.10197
REAL_NU = 1.655e-5 / (LENGTH_SCALE**2 / TIME_SCALE)
```

```python
def read_wf_data(velocity_scale=VELOCITY_SCALE, pressure_scale=PRESSURE_SCALE, data_path=None):
    """Load tiled coordinates and fields, including the
    unvalidated 0.10197 pressure factor. See DATA_PROVENANCE.md before interpreting units.
    """
    path = Path(data_path) if data_path else Path(__file__).resolve().parents[1] / "data_lat.npy"
    if not path.is_file():
        raise FileNotFoundError(f"Missing original data: {path}. Use --smoke-data only for a synthetic execution check.")
    ic = np.load(path, allow_pickle=False).astype(np.float32)
    if ic.ndim != 3 or ic.shape[0] != 3 or not np.isfinite(ic).all():
        raise ValueError("Expected finite upstream data shaped (3, H, W) for u, v, p")
    mesh_y, mesh_x = np.meshgrid(np.linspace(-.720, .719, ic.shape[1]),
                                np.linspace(-.720, .719, ic.shape[2]), indexing="ij")
    xy = np.column_stack((mesh_x.ravel(), mesh_y.ravel())).astype(np.float32)
    fields = np.column_stack((ic[0].ravel() / velocity_scale,
                              ic[1].ravel() / velocity_scale,
                              ic[2].ravel() * LEGACY_PRESSURE_FACTOR / pressure_scale))
    return xy, fields.astype(np.float32)
```

The loader multiplies the array's pressure channel by `0.10197` before scaling it. The units and pressure offset behind this factor are unverified; it is not a valid conversion from pressure to density. Treat this data path as a numerical example until its units are established.

### Step 2: Equations and periodic network

```python
# Use the local equation class defined below.
physics = informer(NavierStokes(nu=nu, rho=1.0, dim=2, time=True), device)
```

```python
class PeriodicFlow(torch.nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.network = mlp(5, 3, cfg)

    def forward(self, xy, t):
        phase = 2 * math.pi * (xy - LOWER) / LENGTH
        # Sine and cosine features enforce spatial periodicity.
        return self.network(torch.cat((phase.sin(), phase.cos(), t), dim=1))
```

```python
class NavierStokes(PDE):
    """Constant-density, 2-D incompressible flow equations."""
    def __init__(self, nu, rho=1.0, dim=2, time=True):
        if dim != 2 or not time:
            raise ValueError("This Lab defines the unsteady 2-D equations only")
        self.dim = 2
        x, y, t = Symbol("x"), Symbol("y"), Symbol("t")
        u, v, p = (Function(name)(x, y, t) for name in ("u", "v", "p"))
        self.equations = {
            "continuity": u.diff(x) + v.diff(y),
            "momentum_x": u.diff(t) + u * u.diff(x) + v * u.diff(y)
                          + p.diff(x) / rho - nu * (u.diff(x, 2) + u.diff(y, 2)),
            "momentum_y": v.diff(t) + u * v.diff(x) + v * v.diff(y)
                          + p.diff(y) / rho - nu * (v.diff(x, 2) + v.diff(y, 2)),
        }
```


### Step 3: Initial data and physics loss

The initial loss compares predicted u, v, and p with the data at t=0, with weight 10 to keep the trivial near-zero flow from dominating early optimization. The interior loss penalizes continuity and momentum residuals. Autograd computes u_t and v_t; PhysicsInformer receives these and computes x/y derivatives from the coordinate tensor. The periodic input features make opposite boundaries match. No positive-time reference values train the network.

```python
def residuals(field, xy, t, physics):
    u, v, p = field.split(1, dim=1)
    return physics.forward({"coordinates": xy, "u": u, "v": v, "p": p,
                            "u__t": derivative(u, t), "v__t": derivative(v, t)})


def loss_terms(model, physics, batch_size, device, initial_data=None):
    xy = (LOWER + LENGTH * torch.rand(batch_size, 2, device=device)).requires_grad_()
    t = torch.rand(batch_size, 1, device=device, requires_grad=True)
    res = residuals(model(xy, t), xy, t, physics)
    if initial_data is None:
        ix = LOWER + LENGTH * torch.rand(batch_size, 2, device=device)
        target = taylor_green(ix, torch.zeros(batch_size, 1, device=device))
    else:
        coords, values = initial_data
        index = torch.randint(len(coords), (batch_size,))
        ix, target = coords[index].to(device), values[index].to(device)
    prediction = model(ix, torch.zeros(batch_size, 1, device=device))
    return {"physics": sum(v.square().mean() for v in res.values()),
            "initial_data": 10.0 * (prediction - target).square().mean()}
```

The full program uses FP32 throughout: 1,000 stochastic Adam updates followed by 2,000 L-BFGS updates on fixed samples (2,048 PDE points and 1,024 initial points). Each line-search evaluation is counted separately in `closure_evaluations`; `STEPS` counts optimizer calls. The architecture and PDE are unchanged, and no analytical solution is built into the learned network.

### Step 4: Evaluate the flow

The synthetic Taylor–Green case provides a known solution for comparison. `heldout_after.per_time` checks a separate 24×24 spatial grid at t = 0, 0.13, 0.37, 0.61, 0.83, and 1. At every time, lesson criteria require velocity RMSE ≤ 0.02, gauge-aligned pressure RMSE ≤ 0.02, and PDE RMSE ≤ 0.05. Initial-field RMSE must be ≤ 0.02; opposite-edge value and spatial-derivative errors must be ≤ 2e-5 and 1e-4. These criteria were set before tuning and apply only to this synthetic teaching example.

Pressure is determined only up to a spatial constant that can vary with time. The evaluator removes the spatial mean pressure error separately at every time, while retaining raw pressure and offset errors for inspection. The optional array has no future reference fields, so assess its initial-data fit and PDE residuals rather than forecast error.

For the array-backed example, normalized time 1 corresponds to 60 hours; the output contains 11 frames at six-hour intervals. The synthetic example uses nondimensional time, not weather-forecast hours.

### Step 5: Choose the data

Use `SMOKE_DATA=True` for Taylor–Green or `SMOKE_DATA=False` for the supplied array. Run the selection cell before training. Both choices train a new model; no pretrained checkpoint is loaded.

[Training program](source_code/navier_stokes.py) · [Configuration](source_code/conf/config.yaml) · [Supplied array](data_lat.npy)

In [ ]:
import os
import sys
import subprocess
import uuid
from pathlib import Path
import numpy as np
from IPython import get_ipython
get_ipython().run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "Start_Here.ipynb").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook inside the bootcamp repository.")
sys.path.insert(0, str(ROOT))
from ETC.runtime.notebook import show_results, validate_settings, completed_output
LAB = ROOT / "01_labs/04_navier_stokes"
OUTPUT_BASE = Path(os.environ.get("AI4SCI_OUTPUT_DIR", str(LAB / "outputs"))).expanduser().resolve()
DEVICE = os.environ.get("AI4SCI_DEVICE", "cpu")
STEPS = int(os.environ.get("AI4SCI_STEPS", "3000"))  # FP32 lesson budget; a short override is only an execution check
RUN_DIRS = {}
RUN_COMPLETED = {}
validate_settings(DEVICE, STEPS)
print("PhysicsNeMo target: 2.2.2", "device:", DEVICE, "steps:", STEPS)

In [ ]:
SMOKE_DATA = os.environ.get("AI4SCI_SMOKE_DATA", "1") == "1"
print("SYNTHETIC Taylor–Green lesson with independent accuracy checks" if SMOKE_DATA else "ORIGINAL data_lat.npy, legacy normalization; not weather validation")
if not SMOKE_DATA:
    original_data = np.load(LAB / "data_lat.npy", mmap_mode="r", allow_pickle=False)
    print("Original data shape:", original_data.shape)

### Step 6: Train and compare

Check the data-selection message above, then run the training cell.

| Selection | Data | Results to inspect |
|---|---|---|
| `SMOKE_DATA=True` | Synthetic Taylor–Green vortex | `synthetic_velocity_rmse`, `synthetic_pressure_gauge_aligned_rmse` and `heldout_after.pde_rmse` |
| `SMOKE_DATA=False` | Supplied `data_lat.npy` array | Initial-data fit and `heldout_after.pde_rmse`; no future-weather targets are available |

For both choices, `data_kind` identifies the source and `initial_data` in `loss.csv` is the initial-field MSE multiplied by 10. The unweighted `heldout_after.initial_data_rmse` and synthetic `accuracy` checks determine whether the lesson criteria were met; loss reduction alone does not. The 3,000-step default uses FP32, including on an L4 GPU; GPU-specific convergence must be measured on that hardware. `weather_forecast_validated` is `False`. See [data provenance](DATA_PROVENANCE.md) for the array's unit limitations.

In [ ]:
RUN_COMPLETED["navier_stokes"] = False
OUTPUT = OUTPUT_BASE / ("navier_stokes-" + uuid.uuid4().hex[:8])
RUN_DIRS["navier_stokes"] = OUTPUT
command = [sys.executable, str(LAB / "source_code/navier_stokes.py"), "--device", DEVICE,
           "--steps", str(STEPS), "--seed", "42", "--output-dir", str(OUTPUT)] + (["--smoke-data"] if SMOKE_DATA else [])
validate_settings(DEVICE, STEPS)
subprocess.run(command, check=True, cwd=ROOT)
RUN_COMPLETED["navier_stokes"] = True
metrics = show_results(OUTPUT, steps=STEPS, seed=42, preview=False)

### Plot the flow

For the synthetic case, compare reference u, predicted u, and their difference at three times. Reference and prediction use the same color range so a near-zero prediction cannot look accurate through automatic rescaling. Pressure accuracy is reported separately after removing each time's spatially constant gauge.

In [ ]:
OUTPUT = completed_output(RUN_DIRS, RUN_COMPLETED, "navier_stokes")
data = np.load(OUTPUT / "predictions.npz", allow_pickle=False)
has_reference = "reference" in data.files
fig, axes = plt.subplots(3 if has_reference else 1, 3, figsize=(13, 11 if has_reference else 4), squeeze=False)
for row, i in enumerate([0, 5, 10]):
    predicted = data["prediction"][i, :, 0]
    if has_reference:
        reference = data["reference"][i, :, 0]
        scale = max(float(np.abs(reference).max()), float(np.abs(predicted).max()), 1e-8)
        panels = [(axes[row, 0], reference, "reference u", scale),
                  (axes[row, 1], predicted, "predicted u", scale),
                  (axes[row, 2], predicted - reference, "u error", max(float(np.abs(predicted - reference).max()), 1e-8))]
    else:
        panels = [(axes[0, row], predicted, "predicted u", max(float(np.abs(predicted).max()), 1e-8))]
    for ax, values, label, scale in panels:
        h = ax.scatter(data["xy"][:, 0], data["xy"][:, 1], c=values, s=6, cmap="coolwarm", vmin=-scale, vmax=scale)
        fig.colorbar(h, ax=ax)
        ax.set(title=f"{label}, t={data['times'][i]:.1f}", xlabel="x", ylabel="y")
fig.suptitle("Synthetic fixture" if metrics["data_kind"] == "synthetic_taylor_green" else "Original initial data; not validated weather prediction")
fig.tight_layout()
plt.show()

## View the flow in ParaView

The next cell exports the final time slice to CSV. In ParaView, apply **Table To Points**, select x/y/z, and color by u or p. All 11 slices remain in `predictions.npz`. The [archived demonstration video](images/paraview.webm) uses VTK files rather than this CSV export.

In [ ]:
OUTPUT = completed_output(RUN_DIRS, RUN_COMPLETED, "navier_stokes")
data = np.load(OUTPUT / "predictions.npz", allow_pickle=False)
xy = data["xy"]
last = data["prediction"][-1]
np.savetxt(OUTPUT / "flow_final.csv", np.column_stack((xy, np.zeros(len(xy)), last)),
           delimiter=",", header="x,y,z,u,v,p", comments="")
print(OUTPUT / "flow_final.csv")

### Next: write the equations yourself

The Labs provided complete programs. In [Wave Level 1](../../02_challenges/01_wave/wave_l1.py), fill in `student_equations` in the Python file, save it, and run the notebook cell. Compare the PDE, initial-condition, and boundary-condition errors with the plot. An unfinished function raises an error identifying the exercise.

Use `USE_REFERENCE=False` to run your code. `USE_REFERENCE=True` selects the instructor equations and trains a new model; it does not load a pretrained answer.

The Challenges cover [Wave](../../02_challenges/01_wave/Challenge_1_Wave_Dynamics.ipynb) (3 levels), [Fluid](../../02_challenges/02_fluid/Challenge_2_Fluid_Flow.ipynb) (3 levels), [Climate](../../02_challenges/03_climate/Challenge_3_Climate_Modeling.ipynb) (2 levels), and [Neural Operators](../../02_challenges/04_neural_operators/Challenge_4_Neural_Operators.ipynb) (3 levels).

[Start Here](../../Start_Here.ipynb) · Previous: [Lab 3: Heat conduction](../03_heat_conduction/Lab_3_Heat_Conduction.ipynb) · Next: [Challenge 1: Wave](../../02_challenges/01_wave/Challenge_1_Wave_Dynamics.ipynb)

--- 

Further reading: [Open Hackathons Resources](https://www.openhackathons.org/s/technical-resources). Community support: [OpenACC and Hackathons Slack Channel](https://www.openacc.org/community#slack).

---

# Licensing

Copyright © 2026 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials may include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.